# Prompt 18: Combining DataFrames — Union, Intersect, Except
## Databricks Certified Associate Developer for Apache Spark
### Topic 3 — DataFrame and Column Operations (30%)

---

## Overview of Set-Based / Vertical Combination Operations

| Method | Duplicates | Column matching | SQL equivalent |
|--------|-----------|-----------------|---------------|
| `union(other)` | **Kept** (no dedup) | By **position** | `UNION ALL` |
| `unionAll(other)` | **Kept** (alias of union) | By **position** | `UNION ALL` |
| `unionByName(other)` | **Kept** (no dedup) | By **name** | `UNION ALL` |
| `intersect(other)` | **Removed** (auto-dedup) | By position | `INTERSECT DISTINCT` |
| `intersectAll(other)` | **Kept** | By position | `INTERSECT ALL` |
| `exceptAll(other)` | **Kept** | By position | `EXCEPT ALL` |
| `subtract(other)` | **Removed** (auto-dedup) | By position | `EXCEPT DISTINCT` |

**Critical exam rule:** `intersect`, `except`, and `subtract` **implicitly deduplicate** the result.
`union`, `unionAll`, and `unionByName` do **NOT** deduplicate — call `.distinct()` if you need uniqueness.

---

## union vs unionByName — Position vs Name Alignment

```python
# union aligns by POSITION — wrong result if column order differs
df_a = spark.createDataFrame([(1, 'Alice')], ['id', 'name'])
df_b = spark.createDataFrame([('Bob', 2)],  ['name', 'id'])   # reversed order!
df_a.union(df_b).show()
# id=1, name='Alice'   ← correct
# id='Bob', name=2     ← WRONG — columns misaligned!

# unionByName aligns by COLUMN NAME — safe when order differs
df_a.unionByName(df_b).show()
# id=1, name='Alice'   ← correct
# id=2, name='Bob'     ← correct — columns matched by name
```

### allowMissingColumns parameter (Spark 3.1+)
```python
# When one DataFrame is missing a column, fill with NULL instead of raising an error
df_with_extra = spark.createDataFrame([(1,'Alice','US')], ['id','name','country'])
df_without    = spark.createDataFrame([(2,'Bob')],        ['id','name'])

# Without allowMissingColumns → AnalysisException: missing column
# df_with_extra.unionByName(df_without)   # ERROR

# With allowMissingColumns=True → missing column filled with NULL
df_with_extra.unionByName(df_without, allowMissingColumns=True).show()
# id=1, name='Alice', country='US'
# id=2, name='Bob',   country=null
```

In [ ]:
# Cell 1: Setup — create sample DataFrames
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import col, lit

spark = SparkSession.builder \
    .appName('CombiningDataFrames') \
    .master('local[*]') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

# January sales
jan = spark.createDataFrame([
    (1, 'Alice',  'Electronics', 1200.0),
    (2, 'Bob',    'Clothing',     450.0),
    (3, 'Carol',  'Electronics',  800.0),
    (4, 'Alice',  'Electronics', 1200.0),  # intentional duplicate of row 1
], ['sale_id', 'customer', 'category', 'amount'])

# February sales
feb = spark.createDataFrame([
    (3, 'Carol',  'Electronics',  800.0),  # same as jan row 3
    (5, 'Dave',   'Books',         95.0),
    (6, 'Eve',    'Clothing',     320.0),
], ['sale_id', 'customer', 'category', 'amount'])

# March sales — columns in different order!
mar_reordered = spark.createDataFrame([
    ('Frank', 7,  550.0, 'Books'),
    ('Alice', 8, 1100.0, 'Electronics'),
], ['customer', 'sale_id', 'amount', 'category'])   # note: different column order

print('=== January ===')
jan.show()
print('=== February ===')
feb.show()
print('=== March (reordered columns) ===')
mar_reordered.show()

In [ ]:
# Cell 2: union and unionByName

print('=== union() — positional alignment, duplicates kept ===')
unioned = jan.union(feb)
print(f'jan rows: {jan.count()}, feb rows: {feb.count()}')
print(f'union rows: {unioned.count()}')  # 4 + 3 = 7 (Carol row appears twice)
unioned.show()

print('=== union() does NOT deduplicate — call .distinct() if needed ===')
unioned_dedup = jan.union(feb).distinct()
print(f'distinct rows after union: {unioned_dedup.count()}')  # 6 unique rows
unioned_dedup.show()

print('=== unionAll() — identical to union() in Spark 2+ ===')
print(f'union rows: {jan.union(feb).count()}')
print(f'unionAll rows: {jan.unionAll(feb).count()}')  # same result

print('=== DANGER: union() with mismatched column order ===')
bad = jan.union(mar_reordered)   # positional alignment — wrong data!
print('Wrong result (sale_id mapped to customer column, etc.):')
bad.show()

print('=== unionByName() — name-based alignment, safe for different column orders ===')
good = jan.unionByName(mar_reordered)
print('Correct result:')
good.show()
print(f'unionByName rows: {good.count()}')  # 4 + 2 = 6

In [ ]:
# Cell 3: unionByName with allowMissingColumns

# Q4 2024 dataset has an extra 'region' column
q4 = spark.createDataFrame([
    (10, 'Grace', 'Electronics', 2200.0, 'APAC'),
    (11, 'Henry', 'Books',        180.0, 'EMEA'),
], ['sale_id', 'customer', 'category', 'amount', 'region'])

# Q3 dataset does NOT have 'region'
q3 = spark.createDataFrame([
    (8, 'Iris', 'Clothing',  430.0),
    (9, 'Jack', 'Electronics', 960.0),
], ['sale_id', 'customer', 'category', 'amount'])

print('=== unionByName without allowMissingColumns — AnalysisException ===')
try:
    q4.unionByName(q3).show()
except Exception as e:
    print(f'Error: {e}')

print('=== unionByName with allowMissingColumns=True — NULL fills missing column ===')
combined = q4.unionByName(q3, allowMissingColumns=True)
combined.show()
combined.printSchema()
print(f'combined rows: {combined.count()}')  # 2 + 2 = 4

print('=== Stacking three monthly datasets with unionByName ===')
all_sales = jan.unionByName(feb).unionByName(mar_reordered)
all_sales.show()
print(f'total rows: {all_sales.count()}')  # 4 + 3 + 2 = 9

In [ ]:
# Cell 4: intersect, intersectAll, except (subtract), exceptAll

# Simpler data for clearer illustration
system_a = spark.createDataFrame([
    (1, 'Alice'),
    (2, 'Bob'),
    (3, 'Carol'),
    (3, 'Carol'),  # duplicate
    (4, 'Dave'),
], ['id', 'name'])

system_b = spark.createDataFrame([
    (3, 'Carol'),
    (3, 'Carol'),  # duplicate
    (4, 'Dave'),
    (5, 'Eve'),
], ['id', 'name'])

print('=== System A ===')
system_a.show()
print('=== System B ===')
system_b.show()

print('=== intersect() — rows in BOTH (auto-dedup) ===')
common = system_a.intersect(system_b)
common.show()                          # Carol(3) and Dave(4) — duplicates collapsed
print(f'intersect count: {common.count()}')  # 2 (not 4 — auto-dedup)

print('=== intersectAll() — rows in BOTH, preserving duplicates ===')
common_all = system_a.intersectAll(system_b)
common_all.show()                      # Carol appears twice (min of 2 and 2 = 2)
print(f'intersectAll count: {common_all.count()}')  # 3 (Carol×2 + Dave×1)

print('=== subtract() — rows in A but NOT in B (auto-dedup, alias of except) ===')
only_a = system_a.subtract(system_b)
only_a.show()                          # Alice(1) and Bob(2) only
print(f'subtract count: {only_a.count()}')  # 2

print('=== exceptAll() — rows in A but NOT in B, preserving duplicates ===')
only_a_all = system_a.exceptAll(system_b)
only_a_all.show()                      # Alice(1) and Bob(2)
print(f'exceptAll count: {only_a_all.count()}')  # 2

print('=== Dedup comparison ===')
print(f'intersect (dedup):    {system_a.intersect(system_b).count()}')
print(f'intersectAll (no dedup): {system_a.intersectAll(system_b).count()}')
print(f'subtract (dedup):     {system_a.subtract(system_b).count()}')
print(f'exceptAll (no dedup): {system_a.exceptAll(system_b).count()}')

In [ ]:
# Cell 5: Practical ETL patterns — stacking, new-record detection, reconciliation

print('=== Pattern 1: Stack 3 monthly CSV-like datasets ===')
# Simulate reading monthly files that arrive with consistent schema
months = [
    spark.createDataFrame([(1,'Alice',100.0,'2024-01-15')], ['id','name','amount','date']),
    spark.createDataFrame([(2,'Bob',  200.0,'2024-02-10')], ['id','name','amount','date']),
    spark.createDataFrame([(3,'Carol',300.0,'2024-03-05')], ['id','name','amount','date']),
]
from functools import reduce
all_months = reduce(lambda a, b: a.unionByName(b), months)
all_months.show()
print(f'Total rows: {all_months.count()}')

print('=== Pattern 2: Detect new records (not yet in warehouse) ===')
# anti-join approach (from Prompt 17) vs subtract approach
incoming = spark.createDataFrame([
    (10, 'New1'),
    (11, 'New2'),
    (1,  'Alice'),   # already exists
    (2,  'Bob'),     # already exists
], ['id', 'name'])

warehouse = spark.createDataFrame([
    (1, 'Alice'),
    (2, 'Bob'),
    (3, 'Carol'),
], ['id', 'name'])

# subtract: rows in incoming not in warehouse
new_records = incoming.subtract(warehouse)
new_records.show()
print(f'New records to insert: {new_records.count()}')  # 2

print('=== Pattern 3: Reconcile — find shared and exclusive records ===')
shared     = incoming.intersect(warehouse)
only_new   = incoming.subtract(warehouse)
only_wh    = warehouse.subtract(incoming)

print(f'In both:               {shared.count()}')   # 2 (Alice, Bob)
print(f'Only in incoming:      {only_new.count()}')  # 2 (New1, New2)
print(f'Only in warehouse:     {only_wh.count()}')   # 1 (Carol)

shared.show()
only_new.show()
only_wh.show()

spark.stop()
print('Done.')

## Real-World Scenarios

<details>
<summary>Scenario 1: Monthly ETL — stack 12 monthly CSVs into one annual DataFrame</summary>

**Situation:**
Twelve monthly CSV files (Jan–Dec) are read into separate DataFrames and need to be combined into a single annual dataset for reporting.

**Code:**
```python
from functools import reduce

# Read each monthly file — schemas match by name but column order may vary
monthly_dfs = [
    spark.read.option('header', True).csv(f'/data/sales/2024-{m:02d}.csv')
    for m in range(1, 13)
]

# unionByName handles any column-order differences between monthly files
annual = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True),
                monthly_dfs)

print(f'Total rows: {annual.count()}')
annual.groupBy('month').count().orderBy('month').show()

annual.write.mode('overwrite').partitionBy('month') \
      .parquet('/warehouse/sales_2024')
```

**Exam Sub-topic:** `unionByName` for name-based alignment; `allowMissingColumns=True`; `reduce` pattern for N DataFrames
</details>

<details>
<summary>Scenario 2: Incremental load — insert only genuinely new records</summary>

**Situation:**
A daily batch delivers customer records. Use `subtract` to find truly new records (not in the warehouse) before inserting.

**Code:**
```python
# Load today's batch
daily_batch = spark.read.parquet('/landing/customers/2024-12-01/')

# Load existing warehouse keys only (project to matching columns for subtract)
existing = spark.read.table('warehouse.customers') \
                .select('customer_id', 'email')  # must match daily_batch schema

# subtract: rows in batch NOT in warehouse (auto-deduplicates)
new_customers = daily_batch.subtract(existing)

print(f'Daily batch:      {daily_batch.count()}')
print(f'Existing:         {existing.count()}')
print(f'Net new to write: {new_customers.count()}')

new_customers.write.mode('append').saveAsTable('warehouse.customers')
```

**Exam Sub-topic:** `subtract` = rows in first not in second; auto-deduplication; ETL incremental load pattern
</details>

<details>
<summary>Scenario 3: System reconciliation — three-way Venn diagram</summary>

**Situation:**
Two CRM systems export customer lists. A reconciliation report must show records in System A only, System B only, and both.

**Code:**
```python
# Each system exports id + email
sys_a = spark.read.csv('/exports/crm_a.csv', header=True)
sys_b = spark.read.csv('/exports/crm_b.csv', header=True)

both    = sys_a.intersect(sys_b)
only_a  = sys_a.subtract(sys_b)
only_b  = sys_b.subtract(sys_a)

print(f'In both systems:    {both.count()}')
print(f'Only in System A:   {only_a.count()}')
print(f'Only in System B:   {only_b.count()}')

# Tag and combine for a report DataFrame
report = both.withColumn('source', lit('Both')) \
    .unionByName(only_a.withColumn('source', lit('A only'))) \
    .unionByName(only_b.withColumn('source', lit('B only')))

report.groupBy('source').count().show()
report.write.mode('overwrite').csv('/reports/reconciliation')
```

**Exam Sub-topic:** `intersect` and `subtract` for set operations; tagging with `lit()` before `unionByName`
</details>

<details>
<summary>Scenario 4: Schema evolution — new column in latest month only</summary>

**Situation:**
Starting in October, the sales export added a `promo_code` column. Earlier months do not have it.
Use `allowMissingColumns=True` to stack all months without errors.

**Code:**
```python
# Months Jan–Sep: no promo_code
old_months = [spark.read.parquet(f'/sales/2024-{m:02d}') for m in range(1, 10)]

# Months Oct–Dec: have promo_code
new_months = [spark.read.parquet(f'/sales/2024-{m:02d}') for m in range(10, 13)]

all_dfs = old_months + new_months

# allowMissingColumns=True fills promo_code with NULL for Jan-Sep
annual = all_dfs[0]
for df in all_dfs[1:]:
    annual = annual.unionByName(df, allowMissingColumns=True)

annual.printSchema()                           # promo_code column present, StringType/nullable
annual.filter(col('promo_code').isNotNull()).count()   # only Oct-Dec rows have values
```

**Exam Sub-topic:** `unionByName(allowMissingColumns=True)` for schema evolution; NULL fills for missing columns
</details>

<details>
<summary>Scenario 5: De-duplication after union across two data sources</summary>

**Situation:**
Two event streams from different regions are unioned. Some events were published to both streams.
Remove duplicates based on `event_id` only (not all columns — one column may differ).

**Code:**
```python
stream_us = spark.read.parquet('/events/us')
stream_eu = spark.read.parquet('/events/eu')

# union includes duplicates
combined = stream_us.union(stream_eu)
print(f'Before dedup: {combined.count()}')

# .distinct() deduplicates on ALL columns — only removes exact duplicates
all_distinct = combined.distinct()
print(f'After distinct: {all_distinct.count()}')

# dropDuplicates(['event_id']) deduplicates on specific key column only
dedup_by_key = combined.dropDuplicates(['event_id'])
print(f'After dropDuplicates on event_id: {dedup_by_key.count()}')

dedup_by_key.write.mode('overwrite').delta('/warehouse/events_deduplicated')
```

**Exam Sub-topic:** `union` does not dedup; `.distinct()` = all-column dedup; `dropDuplicates(['key'])` = column-specific dedup
</details>

## Exam Cheat Sheet

| Question | Answer |
|----------|--------|
| Does `union()` remove duplicates? | **No** — it keeps all rows including duplicates |
| `union` vs `unionAll` | **Identical** in Spark 2+ — `unionAll` is an alias |
| When to use `unionByName` | When DataFrames have the same columns but **in a different order** |
| `allowMissingColumns=True` | Available on `unionByName`; fills missing columns with `NULL` |
| Does `intersect()` remove duplicates? | **Yes** — auto-deduplicates the result |
| `intersect` vs `intersectAll` | `intersect` deduplicates; `intersectAll` preserves duplicates |
| Does `subtract()` remove duplicates? | **Yes** — auto-deduplicates the result |
| `subtract` vs `exceptAll` | `subtract` deduplicates; `exceptAll` preserves duplicates |
| SQL `UNION ALL` equivalent | `union()` or `unionAll()` |
| SQL `UNION DISTINCT` equivalent | `union().distinct()` |
| SQL `INTERSECT ALL` equivalent | `intersectAll()` |
| SQL `EXCEPT DISTINCT` equivalent | `subtract()` |
| How to dedup after union? | Call `.distinct()` or `.dropDuplicates()` after the union |
| `union` column alignment | By **position** — order must match |
| `unionByName` column alignment | By **name** — order does not matter |
| Stack N DataFrames | Use `functools.reduce(lambda a, b: a.unionByName(b), dfs)` |

---

## Practice Q&A

<details>
<summary>Q1: What is the difference between union() and unionByName()?</summary>

**A:**
- **`union(other)`** combines rows from both DataFrames aligned by **column position**. Both DataFrames must have the same number of columns. If the column order differs, data ends up in the wrong columns silently.
- **`unionByName(other)`** aligns columns by **name**. Column order does not matter. Use this when DataFrames may have columns in a different order.
- Neither method removes duplicates. Both are equivalent to SQL `UNION ALL`.
- `unionByName` also accepts `allowMissingColumns=True` (Spark 3.1+) to fill missing columns with `NULL`.
</details>

<details>
<summary>Q2: Which set operations automatically deduplicate the result?</summary>

**A:** The operations that **implicitly deduplicate**:
- `intersect()` — returns unique rows present in both DataFrames
- `subtract()` — returns unique rows in the first DataFrame not in the second

The operations that **preserve duplicates**:
- `union()` / `unionAll()` / `unionByName()` — no dedup; call `.distinct()` if needed
- `intersectAll()` — preserves duplicate occurrences
- `exceptAll()` — preserves duplicate occurrences
</details>

<details>
<summary>Q3: You have DataFrames df1 (columns: id, name) and df2 (columns: name, id). Which combine method should you use?</summary>

**A:** Use **`df1.unionByName(df2)`**.

Using `df1.union(df2)` would produce incorrect results because `union` aligns by position: `df2`'s first column (`name`) would be placed in `df1`'s first column slot (`id`), corrupting the data.

`unionByName` matches columns by name regardless of order, so `id` in df2 correctly maps to `id` in df1.
</details>

<details>
<summary>Q4: You union two DataFrames and call .count(). The result has more rows than expected. Why?</summary>

**A:** `union()` does **not** deduplicate. If the same row appears in both DataFrames, it appears twice in the union result. The count is the sum of both DataFrames' row counts.

To remove duplicates:
- `df1.union(df2).distinct()` — dedup on all columns
- `df1.union(df2).dropDuplicates(['key_col'])` — dedup on specific key column(s)
</details>

<details>
<summary>Q5: How do you combine 12 monthly DataFrames into one without writing a 12-step chain?</summary>

**A:** Use Python's `functools.reduce` to fold the list:

```python
from functools import reduce

monthly_dfs = [jan, feb, mar, apr, may, jun, jul, aug, sep, oct, nov, dec]

annual = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True),
                monthly_dfs)
```

This applies `unionByName` left-to-right across the entire list, handling schema differences via `allowMissingColumns=True`.
</details>